# Decision Tree Regression for Abnormal Returns Prediction

This notebook implements a decision tree regression model to predict abnormal returns based on sentiment scores and tweet volume.

In [2]:
# Import required packages
import pandas as pd
import numpy as np
import os
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error

c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [11]:
# Directories
PROJECT_DIR = r"C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"

CODE_DIR = os.path.join(PROJECT_DIR, "Code")
DATA_DIR = "C:/Users/skazempour/Documents/StockTwits/dataset/v1/data/"
FIGURES_DIR = os.path.join(PROJECT_DIR, "Figures")
TABLES_DIR = os.path.join(PROJECT_DIR, "Tables")
OUTPUT_DATA_DIR = os.path.join(PROJECT_DIR, "Data")

# File names
INPUT_DATA = os.path.join(DATA_DIR, "aggregated", "aggregated_sentiment_returns.pkl") 

In [4]:
data = pd.read_pickle(INPUT_DATA)

In [5]:
# Prepare features and target
# Target variable
TARGET = 'ar_FF5_1'

# Feature engineering
data['log_n_tweets'] = np.log10(1 + data['n_tweets'])

# Select features
FEATURES = ['log_n_tweets', 'sentiment_score']

# Remove missing values
model_data = data[[TARGET] + FEATURES].dropna()

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Features: {FEATURES}")

Sample size: 3,032,691
Target: ar_FF5_1
Features: ['log_n_tweets', 'sentiment_score']


# In-Sample Decision Tree Regression

In [6]:
# Prepare X and y
X = model_data[FEATURES]
y = model_data[TARGET]

# Fit decision tree regression model (in-sample)
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X, y)

# Make predictions
y_pred = dt_model.predict(X)

# Evaluate performance
r2 = r2_score(y, y_pred)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)

print("In-Sample Decision Tree Regression Results")
print("=" * 50)
print(f"R-squared: {r2:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")
print(f"\nTree depth: {dt_model.get_depth()}")
print(f"Number of leaves: {dt_model.get_n_leaves()}")
print(f"\nFeature importances:")
for feature, importance in zip(FEATURES, dt_model.feature_importances_):
    print(f"  {feature}: {importance:.6f}")

In-Sample Decision Tree Regression Results
R-squared: 0.073929
RMSE: 0.052800
MSE: 0.002788

Tree depth: 73
Number of leaves: 51241

Feature importances:
  log_n_tweets: 0.453807
  sentiment_score: 0.546193


# Out-of-Sample Predictions

In [ ]:
# Out-of-sample predictions with multiple window types
# Parameters
TRAIN_END_DATE = '2011-12-31'
WINDOW_252 = 252  # One trading year
WINDOW_21 = 21    # One trading month

# Ensure date column is datetime
model_data['date'] = pd.to_datetime(data.loc[model_data.index, 'date'])

# Sort by date
model_data = model_data.sort_values('date')

# Get unique dates
unique_dates = model_data['date'].unique()
unique_dates = pd.DatetimeIndex(unique_dates).sort_values()

# Find the starting prediction date (first date after training window)
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]

print(f"Training window: {unique_dates[0].strftime('%Y-%m-%d')} to {TRAIN_END_DATE}")
print(f"OOS prediction period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS dates: {len(oos_dates):,}")

In [ ]:
# Initialize storage for predictions
predictions_expanding = []
predictions_rolling_252 = []
predictions_rolling_21 = []

# Loop through OOS dates
for i, pred_date in enumerate(oos_dates):
    # Test data: observations on pred_date
    test_mask = model_data['date'] == pred_date
    X_test = model_data.loc[test_mask, FEATURES]
    
    if len(X_test) == 0:
        continue
    
    test_indices = model_data.index[test_mask]
    
    # 1. Expanding window: all data up to pred_date
    train_mask_exp = model_data['date'] < pred_date
    X_train_exp = model_data.loc[train_mask_exp, FEATURES]
    y_train_exp = model_data.loc[train_mask_exp, TARGET]
    
    if len(X_train_exp) > 0:
        dt_exp = DecisionTreeRegressor(random_state=42)
        dt_exp.fit(X_train_exp, y_train_exp)
        y_pred_exp = dt_exp.predict(X_test)
        
        for idx, pred in zip(test_indices, y_pred_exp):
            predictions_expanding.append({
                'date': pred_date,
                'index': idx,
                'pred_expanding': pred
            })
    
    # 2. Rolling 252-day window
    date_idx = unique_dates.get_loc(pred_date)
    if date_idx >= WINDOW_252:
        start_date_252 = unique_dates[date_idx - WINDOW_252]
        train_mask_252 = (model_data['date'] >= start_date_252) & (model_data['date'] < pred_date)
        X_train_252 = model_data.loc[train_mask_252, FEATURES]
        y_train_252 = model_data.loc[train_mask_252, TARGET]
        
        if len(X_train_252) > 0:
            dt_252 = DecisionTreeRegressor(random_state=42)
            dt_252.fit(X_train_252, y_train_252)
            y_pred_252 = dt_252.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_252):
                predictions_rolling_252.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_252': pred
                })
    
    # 3. Rolling 21-day window
    if date_idx >= WINDOW_21:
        start_date_21 = unique_dates[date_idx - WINDOW_21]
        train_mask_21 = (model_data['date'] >= start_date_21) & (model_data['date'] < pred_date)
        X_train_21 = model_data.loc[train_mask_21, FEATURES]
        y_train_21 = model_data.loc[train_mask_21, TARGET]
        
        if len(X_train_21) > 0:
            dt_21 = DecisionTreeRegressor(random_state=42)
            dt_21.fit(X_train_21, y_train_21)
            y_pred_21 = dt_21.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_21):
                predictions_rolling_21.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_21': pred
                })
    
    # Progress update
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(oos_dates)} dates ({100 * (i + 1) / len(oos_dates):.1f}%)")

print(f"\nCompleted.")
print(f"Expanding window predictions: {len(predictions_expanding):,}")
print(f"Rolling 252-day predictions: {len(predictions_rolling_252):,}")
print(f"Rolling 21-day predictions: {len(predictions_rolling_21):,}")

Training window: 2010-06-02 to 2011-12-31
OOS prediction period: 2012-01-03 to 2024-01-03
Number of OOS dates: 2,948
Processed 100/2948 dates (3.4%)
Processed 200/2948 dates (6.8%)
Processed 300/2948 dates (10.2%)
Processed 400/2948 dates (13.6%)
Processed 500/2948 dates (17.0%)
Processed 600/2948 dates (20.4%)
Processed 700/2948 dates (23.7%)
Processed 800/2948 dates (27.1%)
Processed 900/2948 dates (30.5%)
Processed 1000/2948 dates (33.9%)
Processed 1100/2948 dates (37.3%)
Processed 1200/2948 dates (40.7%)
Processed 1300/2948 dates (44.1%)
Processed 1400/2948 dates (47.5%)
Processed 1500/2948 dates (50.9%)
Processed 1600/2948 dates (54.3%)
Processed 1700/2948 dates (57.7%)
Processed 1800/2948 dates (61.1%)
Processed 1900/2948 dates (64.5%)
Processed 2000/2948 dates (67.8%)
Processed 2100/2948 dates (71.2%)
Processed 2200/2948 dates (74.6%)
Processed 2300/2948 dates (78.0%)
Processed 2400/2948 dates (81.4%)
Processed 2500/2948 dates (84.8%)
Processed 2600/2948 dates (88.2%)
Processed 

In [8]:
# Convert predictions to DataFrames and merge
df_expanding = pd.DataFrame(predictions_expanding)
df_rolling_252 = pd.DataFrame(predictions_rolling_252)
df_rolling_21 = pd.DataFrame(predictions_rolling_21)

# Merge all predictions together
predictions_df = df_expanding.merge(
    df_rolling_252,
    on=['date', 'index'],
    how='outer'
).merge(
    df_rolling_21,
    on=['date', 'index'],
    how='outer'
)

# Add symbol information
predictions_df = predictions_df.merge(
    data[['symbol']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder columns
predictions_df = predictions_df[['date', 'symbol', 'index', 'pred_expanding', 'pred_rolling_252', 'pred_rolling_21']]

# Sort by date and symbol
predictions_df = predictions_df.sort_values(['date', 'symbol']).reset_index(drop=True)

print("Predictions Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_df):,}")
print(f"\nNon-null predictions by window type:")
print(f"  Expanding: {predictions_df['pred_expanding'].notna().sum():,}")
print(f"  Rolling 252-day: {predictions_df['pred_rolling_252'].notna().sum():,}")
print(f"  Rolling 21-day: {predictions_df['pred_rolling_21'].notna().sum():,}")
print(f"\nFirst 10 predictions:")
print(predictions_df.head(10))

Predictions Summary
Total observations: 3,001,267

Non-null predictions by window type:
  Expanding: 3,001,267
  Rolling 252-day: 3,001,267
  Rolling 21-day: 3,001,267

First 10 predictions:
        date symbol  index  pred_expanding  pred_rolling_252  pred_rolling_21
0 2012-01-03   AAPL  31505       -0.002871         -0.005829         0.005926
1 2012-01-03    AAV  31757       -0.000163         -0.000440        -0.000325
2 2012-01-03   ABVT  31838       -0.000163         -0.000440        -0.000325
3 2012-01-03    ACI  31993       -0.000163         -0.000440        -0.000325
4 2012-01-03    ACW  32078       -0.001678         -0.001156        -0.002704
5 2012-01-03    AKS  32643       -0.000163         -0.000440        -0.000325
6 2012-01-03    ALK  32752       -0.000163         -0.000440        -0.000325
7 2012-01-03   ALXN  32881       -0.000262          0.000002         0.004386
8 2012-01-03     AM  32942        0.000391          0.000409        -0.001524
9 2012-01-03    AME  33040   

In [ ]:
# Save predictions to Data directory
os.makedirs(OUTPUT_DATA_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(OUTPUT_DATA_DIR, "predictions_decision_tree.pkl")
predictions_df.to_pickle(OUTPUT_FILE)

print(f"Predictions saved to: {OUTPUT_FILE}")
print(f"File size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.2f} MB")
print(f"Shape: {predictions_df.shape}")

Predictions saved to: C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/Data\predictions_decision_tree.pkl
File size: 128.85 MB
Shape: (3001267, 6)


# Regularized Decision Trees - Out-of-Sample Predictions

## Method 1: Shallow Tree (max_depth=5)

In [9]:
# Method 1: Shallow Tree (max_depth=5)
print("=" * 70)
print("METHOD 1: SHALLOW TREE (max_depth=5)")
print("=" * 70)

# Initialize storage for predictions
predictions_shallow_exp = []
predictions_shallow_252 = []
predictions_shallow_21 = []

# Loop through OOS dates
for i, pred_date in enumerate(oos_dates):
    # Test data: observations on pred_date
    test_mask = model_data['date'] == pred_date
    X_test = model_data.loc[test_mask, FEATURES]
    
    if len(X_test) == 0:
        continue
    
    test_indices = model_data.index[test_mask]
    
    # 1. Expanding window
    train_mask_exp = model_data['date'] < pred_date
    X_train_exp = model_data.loc[train_mask_exp, FEATURES]
    y_train_exp = model_data.loc[train_mask_exp, TARGET]
    
    if len(X_train_exp) > 0:
        dt_shallow = DecisionTreeRegressor(max_depth=5, random_state=42)
        dt_shallow.fit(X_train_exp, y_train_exp)
        y_pred_exp = dt_shallow.predict(X_test)
        
        for idx, pred in zip(test_indices, y_pred_exp):
            predictions_shallow_exp.append({
                'date': pred_date,
                'index': idx,
                'pred_expanding': pred
            })
    
    # 2. Rolling 252-day window
    date_idx = unique_dates.get_loc(pred_date)
    if date_idx >= WINDOW_252:
        start_date_252 = unique_dates[date_idx - WINDOW_252]
        train_mask_252 = (model_data['date'] >= start_date_252) & (model_data['date'] < pred_date)
        X_train_252 = model_data.loc[train_mask_252, FEATURES]
        y_train_252 = model_data.loc[train_mask_252, TARGET]
        
        if len(X_train_252) > 0:
            dt_shallow = DecisionTreeRegressor(max_depth=5, random_state=42)
            dt_shallow.fit(X_train_252, y_train_252)
            y_pred_252 = dt_shallow.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_252):
                predictions_shallow_252.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_252': pred
                })
    
    # 3. Rolling 21-day window
    if date_idx >= WINDOW_21:
        start_date_21 = unique_dates[date_idx - WINDOW_21]
        train_mask_21 = (model_data['date'] >= start_date_21) & (model_data['date'] < pred_date)
        X_train_21 = model_data.loc[train_mask_21, FEATURES]
        y_train_21 = model_data.loc[train_mask_21, TARGET]
        
        if len(X_train_21) > 0:
            dt_shallow = DecisionTreeRegressor(max_depth=5, random_state=42)
            dt_shallow.fit(X_train_21, y_train_21)
            y_pred_21 = dt_shallow.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_21):
                predictions_shallow_21.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_21': pred
                })
    
    # Progress update
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(oos_dates)} dates ({100 * (i + 1) / len(oos_dates):.1f}%)")

print(f"\nCompleted.")
print(f"Expanding window predictions: {len(predictions_shallow_exp):,}")
print(f"Rolling 252-day predictions: {len(predictions_shallow_252):,}")
print(f"Rolling 21-day predictions: {len(predictions_shallow_21):,}")

METHOD 1: SHALLOW TREE (max_depth=5)
Processed 100/2948 dates (3.4%)
Processed 100/2948 dates (3.4%)
Processed 200/2948 dates (6.8%)
Processed 200/2948 dates (6.8%)
Processed 300/2948 dates (10.2%)
Processed 300/2948 dates (10.2%)
Processed 400/2948 dates (13.6%)
Processed 400/2948 dates (13.6%)
Processed 500/2948 dates (17.0%)
Processed 500/2948 dates (17.0%)
Processed 600/2948 dates (20.4%)
Processed 600/2948 dates (20.4%)
Processed 700/2948 dates (23.7%)
Processed 700/2948 dates (23.7%)
Processed 800/2948 dates (27.1%)
Processed 800/2948 dates (27.1%)
Processed 900/2948 dates (30.5%)
Processed 900/2948 dates (30.5%)
Processed 1000/2948 dates (33.9%)
Processed 1000/2948 dates (33.9%)
Processed 1100/2948 dates (37.3%)
Processed 1100/2948 dates (37.3%)
Processed 1200/2948 dates (40.7%)
Processed 1200/2948 dates (40.7%)
Processed 1300/2948 dates (44.1%)
Processed 1300/2948 dates (44.1%)
Processed 1400/2948 dates (47.5%)
Processed 1400/2948 dates (47.5%)
Processed 1500/2948 dates (50.9%)

In [12]:
# Merge and save Method 1 predictions
df_shallow_exp = pd.DataFrame(predictions_shallow_exp)
df_shallow_252 = pd.DataFrame(predictions_shallow_252)
df_shallow_21 = pd.DataFrame(predictions_shallow_21)

predictions_shallow = df_shallow_exp.merge(
    df_shallow_252,
    on=['date', 'index'],
    how='outer'
).merge(
    df_shallow_21,
    on=['date', 'index'],
    how='outer'
)

# Add symbol information
predictions_shallow = predictions_shallow.merge(
    data[['symbol']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder and sort
predictions_shallow = predictions_shallow[['date', 'symbol', 'index', 'pred_expanding', 'pred_rolling_252', 'pred_rolling_21']]
predictions_shallow = predictions_shallow.sort_values(['date', 'symbol']).reset_index(drop=True)

# Save predictions
OUTPUT_FILE_SHALLOW = os.path.join(OUTPUT_DATA_DIR, "predictions_decision_tree_shallow.pkl")
predictions_shallow.to_pickle(OUTPUT_FILE_SHALLOW)

print("\nMethod 1 - Shallow Tree Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_shallow):,}")
print(f"Non-null predictions by window type:")
print(f"  Expanding: {predictions_shallow['pred_expanding'].notna().sum():,}")
print(f"  Rolling 252-day: {predictions_shallow['pred_rolling_252'].notna().sum():,}")
print(f"  Rolling 21-day: {predictions_shallow['pred_rolling_21'].notna().sum():,}")
print(f"\nSaved to: {OUTPUT_FILE_SHALLOW}")
print(f"File size: {os.path.getsize(OUTPUT_FILE_SHALLOW) / (1024**2):.2f} MB")


Method 1 - Shallow Tree Summary
Total observations: 3,001,267
Non-null predictions by window type:
  Expanding: 3,001,267
  Rolling 252-day: 3,001,267
  Rolling 21-day: 3,001,267

Saved to: C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/Data\predictions_decision_tree_shallow.pkl
File size: 128.85 MB


## Method 2: Moderate Tree (max_depth=10, min_samples_leaf=50)

In [13]:
# Method 2: Moderate Tree (max_depth=10, min_samples_leaf=50)
print("=" * 70)
print("METHOD 2: MODERATE TREE (max_depth=10, min_samples_leaf=50)")
print("=" * 70)

# Initialize storage for predictions
predictions_moderate_exp = []
predictions_moderate_252 = []
predictions_moderate_21 = []

# Loop through OOS dates
for i, pred_date in enumerate(oos_dates):
    # Test data: observations on pred_date
    test_mask = model_data['date'] == pred_date
    X_test = model_data.loc[test_mask, FEATURES]
    
    if len(X_test) == 0:
        continue
    
    test_indices = model_data.index[test_mask]
    
    # 1. Expanding window
    train_mask_exp = model_data['date'] < pred_date
    X_train_exp = model_data.loc[train_mask_exp, FEATURES]
    y_train_exp = model_data.loc[train_mask_exp, TARGET]
    
    if len(X_train_exp) > 0:
        dt_moderate = DecisionTreeRegressor(max_depth=10, min_samples_leaf=50, random_state=42)
        dt_moderate.fit(X_train_exp, y_train_exp)
        y_pred_exp = dt_moderate.predict(X_test)
        
        for idx, pred in zip(test_indices, y_pred_exp):
            predictions_moderate_exp.append({
                'date': pred_date,
                'index': idx,
                'pred_expanding': pred
            })
    
    # 2. Rolling 252-day window
    date_idx = unique_dates.get_loc(pred_date)
    if date_idx >= WINDOW_252:
        start_date_252 = unique_dates[date_idx - WINDOW_252]
        train_mask_252 = (model_data['date'] >= start_date_252) & (model_data['date'] < pred_date)
        X_train_252 = model_data.loc[train_mask_252, FEATURES]
        y_train_252 = model_data.loc[train_mask_252, TARGET]
        
        if len(X_train_252) > 0:
            dt_moderate = DecisionTreeRegressor(max_depth=10, min_samples_leaf=50, random_state=42)
            dt_moderate.fit(X_train_252, y_train_252)
            y_pred_252 = dt_moderate.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_252):
                predictions_moderate_252.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_252': pred
                })
    
    # 3. Rolling 21-day window
    if date_idx >= WINDOW_21:
        start_date_21 = unique_dates[date_idx - WINDOW_21]
        train_mask_21 = (model_data['date'] >= start_date_21) & (model_data['date'] < pred_date)
        X_train_21 = model_data.loc[train_mask_21, FEATURES]
        y_train_21 = model_data.loc[train_mask_21, TARGET]
        
        if len(X_train_21) > 0:
            dt_moderate = DecisionTreeRegressor(max_depth=10, min_samples_leaf=50, random_state=42)
            dt_moderate.fit(X_train_21, y_train_21)
            y_pred_21 = dt_moderate.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_21):
                predictions_moderate_21.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_21': pred
                })
    
    # Progress update
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(oos_dates)} dates ({100 * (i + 1) / len(oos_dates):.1f}%)")

print(f"\nCompleted.")
print(f"Expanding window predictions: {len(predictions_moderate_exp):,}")
print(f"Rolling 252-day predictions: {len(predictions_moderate_252):,}")
print(f"Rolling 21-day predictions: {len(predictions_moderate_21):,}")

METHOD 2: MODERATE TREE (max_depth=10, min_samples_leaf=50)
Processed 100/2948 dates (3.4%)
Processed 100/2948 dates (3.4%)
Processed 200/2948 dates (6.8%)
Processed 200/2948 dates (6.8%)
Processed 300/2948 dates (10.2%)
Processed 300/2948 dates (10.2%)
Processed 400/2948 dates (13.6%)
Processed 400/2948 dates (13.6%)
Processed 500/2948 dates (17.0%)
Processed 500/2948 dates (17.0%)
Processed 600/2948 dates (20.4%)
Processed 600/2948 dates (20.4%)
Processed 700/2948 dates (23.7%)
Processed 700/2948 dates (23.7%)
Processed 800/2948 dates (27.1%)
Processed 800/2948 dates (27.1%)
Processed 900/2948 dates (30.5%)
Processed 900/2948 dates (30.5%)
Processed 1000/2948 dates (33.9%)
Processed 1000/2948 dates (33.9%)
Processed 1100/2948 dates (37.3%)
Processed 1100/2948 dates (37.3%)
Processed 1200/2948 dates (40.7%)
Processed 1200/2948 dates (40.7%)
Processed 1300/2948 dates (44.1%)
Processed 1300/2948 dates (44.1%)
Processed 1400/2948 dates (47.5%)
Processed 1400/2948 dates (47.5%)
Processed 

In [14]:
# Merge and save Method 2 predictions
df_moderate_exp = pd.DataFrame(predictions_moderate_exp)
df_moderate_252 = pd.DataFrame(predictions_moderate_252)
df_moderate_21 = pd.DataFrame(predictions_moderate_21)

predictions_moderate = df_moderate_exp.merge(
    df_moderate_252,
    on=['date', 'index'],
    how='outer'
).merge(
    df_moderate_21,
    on=['date', 'index'],
    how='outer'
)

# Add symbol information
predictions_moderate = predictions_moderate.merge(
    data[['symbol']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder and sort
predictions_moderate = predictions_moderate[['date', 'symbol', 'index', 'pred_expanding', 'pred_rolling_252', 'pred_rolling_21']]
predictions_moderate = predictions_moderate.sort_values(['date', 'symbol']).reset_index(drop=True)

# Save predictions
OUTPUT_FILE_MODERATE = os.path.join(OUTPUT_DATA_DIR, "predictions_decision_tree_moderate.pkl")
predictions_moderate.to_pickle(OUTPUT_FILE_MODERATE)

print("\nMethod 2 - Moderate Tree Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_moderate):,}")
print(f"Non-null predictions by window type:")
print(f"  Expanding: {predictions_moderate['pred_expanding'].notna().sum():,}")
print(f"  Rolling 252-day: {predictions_moderate['pred_rolling_252'].notna().sum():,}")
print(f"  Rolling 21-day: {predictions_moderate['pred_rolling_21'].notna().sum():,}")
print(f"\nSaved to: {OUTPUT_FILE_MODERATE}")
print(f"File size: {os.path.getsize(OUTPUT_FILE_MODERATE) / (1024**2):.2f} MB")


Method 2 - Moderate Tree Summary
Total observations: 3,001,267
Non-null predictions by window type:
  Expanding: 3,001,267
  Rolling 252-day: 3,001,267
  Rolling 21-day: 3,001,267

Saved to: C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/Data\predictions_decision_tree_moderate.pkl
File size: 128.85 MB


## Method 3: Constrained Tree (max_leaf_nodes=50, min_samples_split=100)

In [15]:
# Method 3: Constrained Tree (max_leaf_nodes=50, min_samples_split=100)
print("=" * 70)
print("METHOD 3: CONSTRAINED TREE (max_leaf_nodes=50, min_samples_split=100)")
print("=" * 70)

# Initialize storage for predictions
predictions_constrained_exp = []
predictions_constrained_252 = []
predictions_constrained_21 = []

# Loop through OOS dates
for i, pred_date in enumerate(oos_dates):
    # Test data: observations on pred_date
    test_mask = model_data['date'] == pred_date
    X_test = model_data.loc[test_mask, FEATURES]
    
    if len(X_test) == 0:
        continue
    
    test_indices = model_data.index[test_mask]
    
    # 1. Expanding window
    train_mask_exp = model_data['date'] < pred_date
    X_train_exp = model_data.loc[train_mask_exp, FEATURES]
    y_train_exp = model_data.loc[train_mask_exp, TARGET]
    
    if len(X_train_exp) > 0:
        dt_constrained = DecisionTreeRegressor(max_leaf_nodes=50, min_samples_split=100, random_state=42)
        dt_constrained.fit(X_train_exp, y_train_exp)
        y_pred_exp = dt_constrained.predict(X_test)
        
        for idx, pred in zip(test_indices, y_pred_exp):
            predictions_constrained_exp.append({
                'date': pred_date,
                'index': idx,
                'pred_expanding': pred
            })
    
    # 2. Rolling 252-day window
    date_idx = unique_dates.get_loc(pred_date)
    if date_idx >= WINDOW_252:
        start_date_252 = unique_dates[date_idx - WINDOW_252]
        train_mask_252 = (model_data['date'] >= start_date_252) & (model_data['date'] < pred_date)
        X_train_252 = model_data.loc[train_mask_252, FEATURES]
        y_train_252 = model_data.loc[train_mask_252, TARGET]
        
        if len(X_train_252) > 0:
            dt_constrained = DecisionTreeRegressor(max_leaf_nodes=50, min_samples_split=100, random_state=42)
            dt_constrained.fit(X_train_252, y_train_252)
            y_pred_252 = dt_constrained.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_252):
                predictions_constrained_252.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_252': pred
                })
    
    # 3. Rolling 21-day window
    if date_idx >= WINDOW_21:
        start_date_21 = unique_dates[date_idx - WINDOW_21]
        train_mask_21 = (model_data['date'] >= start_date_21) & (model_data['date'] < pred_date)
        X_train_21 = model_data.loc[train_mask_21, FEATURES]
        y_train_21 = model_data.loc[train_mask_21, TARGET]
        
        if len(X_train_21) > 0:
            dt_constrained = DecisionTreeRegressor(max_leaf_nodes=50, min_samples_split=100, random_state=42)
            dt_constrained.fit(X_train_21, y_train_21)
            y_pred_21 = dt_constrained.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_21):
                predictions_constrained_21.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_21': pred
                })
    
    # Progress update
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(oos_dates)} dates ({100 * (i + 1) / len(oos_dates):.1f}%)")

print(f"\nCompleted.")
print(f"Expanding window predictions: {len(predictions_constrained_exp):,}")
print(f"Rolling 252-day predictions: {len(predictions_constrained_252):,}")
print(f"Rolling 21-day predictions: {len(predictions_constrained_21):,}")

METHOD 3: CONSTRAINED TREE (max_leaf_nodes=50, min_samples_split=100)
Processed 100/2948 dates (3.4%)
Processed 100/2948 dates (3.4%)
Processed 200/2948 dates (6.8%)
Processed 200/2948 dates (6.8%)
Processed 300/2948 dates (10.2%)
Processed 300/2948 dates (10.2%)
Processed 400/2948 dates (13.6%)
Processed 400/2948 dates (13.6%)
Processed 500/2948 dates (17.0%)
Processed 500/2948 dates (17.0%)
Processed 600/2948 dates (20.4%)
Processed 600/2948 dates (20.4%)
Processed 700/2948 dates (23.7%)
Processed 700/2948 dates (23.7%)
Processed 800/2948 dates (27.1%)
Processed 800/2948 dates (27.1%)
Processed 900/2948 dates (30.5%)
Processed 900/2948 dates (30.5%)
Processed 1000/2948 dates (33.9%)
Processed 1000/2948 dates (33.9%)
Processed 1100/2948 dates (37.3%)
Processed 1100/2948 dates (37.3%)
Processed 1200/2948 dates (40.7%)
Processed 1200/2948 dates (40.7%)
Processed 1300/2948 dates (44.1%)
Processed 1300/2948 dates (44.1%)
Processed 1400/2948 dates (47.5%)
Processed 1400/2948 dates (47.5%)


In [16]:
# Merge and save Method 3 predictions
df_constrained_exp = pd.DataFrame(predictions_constrained_exp)
df_constrained_252 = pd.DataFrame(predictions_constrained_252)
df_constrained_21 = pd.DataFrame(predictions_constrained_21)

predictions_constrained = df_constrained_exp.merge(
    df_constrained_252,
    on=['date', 'index'],
    how='outer'
).merge(
    df_constrained_21,
    on=['date', 'index'],
    how='outer'
)

# Add symbol information
predictions_constrained = predictions_constrained.merge(
    data[['symbol']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder and sort
predictions_constrained = predictions_constrained[['date', 'symbol', 'index', 'pred_expanding', 'pred_rolling_252', 'pred_rolling_21']]
predictions_constrained = predictions_constrained.sort_values(['date', 'symbol']).reset_index(drop=True)

# Save predictions
OUTPUT_FILE_CONSTRAINED = os.path.join(OUTPUT_DATA_DIR, "predictions_decision_tree_constrained.pkl")
predictions_constrained.to_pickle(OUTPUT_FILE_CONSTRAINED)

print("\nMethod 3 - Constrained Tree Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_constrained):,}")
print(f"Non-null predictions by window type:")
print(f"  Expanding: {predictions_constrained['pred_expanding'].notna().sum():,}")
print(f"  Rolling 252-day: {predictions_constrained['pred_rolling_252'].notna().sum():,}")
print(f"  Rolling 21-day: {predictions_constrained['pred_rolling_21'].notna().sum():,}")
print(f"\nSaved to: {OUTPUT_FILE_CONSTRAINED}")
print(f"File size: {os.path.getsize(OUTPUT_FILE_CONSTRAINED) / (1024**2):.2f} MB")


Method 3 - Constrained Tree Summary
Total observations: 3,001,267
Non-null predictions by window type:
  Expanding: 3,001,267
  Rolling 252-day: 3,001,267
  Rolling 21-day: 3,001,267

Saved to: C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/Data\predictions_decision_tree_constrained.pkl
File size: 128.85 MB


## Method 4: Post-Pruned Tree (Cost-Complexity Pruning)

In [ ]:
# Method 4: Post-Pruned Tree using Cost-Complexity Pruning (ccp_alpha)
# We'll use a moderate ccp_alpha value that provides good regularization
# Typical values range from 0.0001 to 0.01 depending on the dataset

print("=" * 70)
print("METHOD 4: POST-PRUNED TREE (ccp_alpha=0.001)")
print("=" * 70)

# Initialize storage for predictions
predictions_pruned_exp = []
predictions_pruned_252 = []
predictions_pruned_21 = []

# Loop through OOS dates
for i, pred_date in enumerate(oos_dates):
    # Test data: observations on pred_date
    test_mask = model_data['date'] == pred_date
    X_test = model_data.loc[test_mask, FEATURES]
    
    if len(X_test) == 0:
        continue
    
    test_indices = model_data.index[test_mask]
    
    # 1. Expanding window
    train_mask_exp = model_data['date'] < pred_date
    X_train_exp = model_data.loc[train_mask_exp, FEATURES]
    y_train_exp = model_data.loc[train_mask_exp, TARGET]
    
    if len(X_train_exp) > 0:
        dt_pruned = DecisionTreeRegressor(ccp_alpha=0.001, random_state=42)
        dt_pruned.fit(X_train_exp, y_train_exp)
        y_pred_exp = dt_pruned.predict(X_test)
        
        for idx, pred in zip(test_indices, y_pred_exp):
            predictions_pruned_exp.append({
                'date': pred_date,
                'index': idx,
                'pred_expanding': pred
            })
    
    # 2. Rolling 252-day window
    date_idx = unique_dates.get_loc(pred_date)
    if date_idx >= WINDOW_252:
        start_date_252 = unique_dates[date_idx - WINDOW_252]
        train_mask_252 = (model_data['date'] >= start_date_252) & (model_data['date'] < pred_date)
        X_train_252 = model_data.loc[train_mask_252, FEATURES]
        y_train_252 = model_data.loc[train_mask_252, TARGET]
        
        if len(X_train_252) > 0:
            dt_pruned = DecisionTreeRegressor(ccp_alpha=0.001, random_state=42)
            dt_pruned.fit(X_train_252, y_train_252)
            y_pred_252 = dt_pruned.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_252):
                predictions_pruned_252.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_252': pred
                })
    
    # 3. Rolling 21-day window
    if date_idx >= WINDOW_21:
        start_date_21 = unique_dates[date_idx - WINDOW_21]
        train_mask_21 = (model_data['date'] >= start_date_21) & (model_data['date'] < pred_date)
        X_train_21 = model_data.loc[train_mask_21, FEATURES]
        y_train_21 = model_data.loc[train_mask_21, TARGET]
        
        if len(X_train_21) > 0:
            dt_pruned = DecisionTreeRegressor(ccp_alpha=0.001, random_state=42)
            dt_pruned.fit(X_train_21, y_train_21)
            y_pred_21 = dt_pruned.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_21):
                predictions_pruned_21.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_21': pred
                })
    
    # Progress update
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(oos_dates)} dates ({100 * (i + 1) / len(oos_dates):.1f}%)")

print(f"\nCompleted.")
print(f"Expanding window predictions: {len(predictions_pruned_exp):,}")
print(f"Rolling 252-day predictions: {len(predictions_pruned_252):,}")
print(f"Rolling 21-day predictions: {len(predictions_pruned_21):,}")

In [ ]:
# Merge and save Method 4 predictions
df_pruned_exp = pd.DataFrame(predictions_pruned_exp)
df_pruned_252 = pd.DataFrame(predictions_pruned_252)
df_pruned_21 = pd.DataFrame(predictions_pruned_21)

predictions_pruned = df_pruned_exp.merge(
    df_pruned_252,
    on=['date', 'index'],
    how='outer'
).merge(
    df_pruned_21,
    on=['date', 'index'],
    how='outer'
)

# Add symbol information
predictions_pruned = predictions_pruned.merge(
    data[['symbol']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder and sort
predictions_pruned = predictions_pruned[['date', 'symbol', 'index', 'pred_expanding', 'pred_rolling_252', 'pred_rolling_21']]
predictions_pruned = predictions_pruned.sort_values(['date', 'symbol']).reset_index(drop=True)

# Save predictions
OUTPUT_FILE_PRUNED = os.path.join(OUTPUT_DATA_DIR, "predictions_decision_tree_pruned.pkl")
predictions_pruned.to_pickle(OUTPUT_FILE_PRUNED)

print("\nMethod 4 - Post-Pruned Tree Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_pruned):,}")
print(f"Non-null predictions by window type:")
print(f"  Expanding: {predictions_pruned['pred_expanding'].notna().sum():,}")
print(f"  Rolling 252-day: {predictions_pruned['pred_rolling_252'].notna().sum():,}")
print(f"  Rolling 21-day: {predictions_pruned['pred_rolling_21'].notna().sum():,}")
print(f"\nSaved to: {OUTPUT_FILE_PRUNED}")
print(f"File size: {os.path.getsize(OUTPUT_FILE_PRUNED) / (1024**2):.2f} MB")

## Method 5: Optimal Post-Pruned Tree (CV-Selected ccp_alpha)

In [17]:
# Method 5: Post-Pruned Tree with Cross-Validation to select optimal ccp_alpha
# For each training window, we'll find the optimal alpha via CV and use it for prediction

from sklearn.model_selection import cross_val_score

print("=" * 70)
print("METHOD 5: CV-OPTIMIZED POST-PRUNED TREE")
print("=" * 70)

# Initialize storage for predictions and alpha values
predictions_cv_pruned_exp = []
predictions_cv_pruned_252 = []
predictions_cv_pruned_21 = []
alpha_history = []

# Loop through OOS dates
for i, pred_date in enumerate(oos_dates):
    # Test data: observations on pred_date
    test_mask = model_data['date'] == pred_date
    X_test = model_data.loc[test_mask, FEATURES]
    
    if len(X_test) == 0:
        continue
    
    test_indices = model_data.index[test_mask]
    
    # 1. Expanding window
    train_mask_exp = model_data['date'] < pred_date
    X_train_exp = model_data.loc[train_mask_exp, FEATURES]
    y_train_exp = model_data.loc[train_mask_exp, TARGET]
    
    if len(X_train_exp) > 100:  # Need sufficient data for CV
        # Get pruning path
        dt_temp = DecisionTreeRegressor(random_state=42)
        path = dt_temp.cost_complexity_pruning_path(X_train_exp, y_train_exp)
        ccp_alphas = path.ccp_alphas
        
        # Sample alphas to test (to speed up CV)
        # Test every 10th alpha or max 20 values
        step = max(1, len(ccp_alphas) // 20)
        alphas_to_test = ccp_alphas[::step]
        
        # Cross-validate to find best alpha
        best_alpha = 0
        best_score = -np.inf
        
        for alpha in alphas_to_test:
            if alpha == 0:
                continue
            dt_cv = DecisionTreeRegressor(ccp_alpha=alpha, random_state=42)
            # Use 5-fold CV with negative MSE
            scores = cross_val_score(dt_cv, X_train_exp, y_train_exp, 
                                    cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
            mean_score = scores.mean()
            
            if mean_score > best_score:
                best_score = mean_score
                best_alpha = alpha
        
        # Use best alpha if found, otherwise use small default
        alpha_to_use = best_alpha if best_alpha > 0 else 0.0001
        
        # Train final model with best alpha
        dt_cv_pruned = DecisionTreeRegressor(ccp_alpha=alpha_to_use, random_state=42)
        dt_cv_pruned.fit(X_train_exp, y_train_exp)
        y_pred_exp = dt_cv_pruned.predict(X_test)
        
        for idx, pred in zip(test_indices, y_pred_exp):
            predictions_cv_pruned_exp.append({
                'date': pred_date,
                'index': idx,
                'pred_expanding': pred
            })
        
        alpha_history.append({
            'date': pred_date,
            'window': 'expanding',
            'alpha': alpha_to_use,
            'n_train': len(X_train_exp)
        })
    
    # 2. Rolling 252-day window
    date_idx = unique_dates.get_loc(pred_date)
    if date_idx >= WINDOW_252:
        start_date_252 = unique_dates[date_idx - WINDOW_252]
        train_mask_252 = (model_data['date'] >= start_date_252) & (model_data['date'] < pred_date)
        X_train_252 = model_data.loc[train_mask_252, FEATURES]
        y_train_252 = model_data.loc[train_mask_252, TARGET]
        
        if len(X_train_252) > 100:
            # Get pruning path
            dt_temp = DecisionTreeRegressor(random_state=42)
            path = dt_temp.cost_complexity_pruning_path(X_train_252, y_train_252)
            ccp_alphas = path.ccp_alphas
            
            step = max(1, len(ccp_alphas) // 20)
            alphas_to_test = ccp_alphas[::step]
            
            best_alpha = 0
            best_score = -np.inf
            
            for alpha in alphas_to_test:
                if alpha == 0:
                    continue
                dt_cv = DecisionTreeRegressor(ccp_alpha=alpha, random_state=42)
                scores = cross_val_score(dt_cv, X_train_252, y_train_252, 
                                        cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
                mean_score = scores.mean()
                
                if mean_score > best_score:
                    best_score = mean_score
                    best_alpha = alpha
            
            alpha_to_use = best_alpha if best_alpha > 0 else 0.0001
            
            dt_cv_pruned = DecisionTreeRegressor(ccp_alpha=alpha_to_use, random_state=42)
            dt_cv_pruned.fit(X_train_252, y_train_252)
            y_pred_252 = dt_cv_pruned.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_252):
                predictions_cv_pruned_252.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_252': pred
                })
    
    # 3. Rolling 21-day window
    if date_idx >= WINDOW_21:
        start_date_21 = unique_dates[date_idx - WINDOW_21]
        train_mask_21 = (model_data['date'] >= start_date_21) & (model_data['date'] < pred_date)
        X_train_21 = model_data.loc[train_mask_21, FEATURES]
        y_train_21 = model_data.loc[train_mask_21, TARGET]
        
        if len(X_train_21) > 100:
            # Get pruning path
            dt_temp = DecisionTreeRegressor(random_state=42)
            path = dt_temp.cost_complexity_pruning_path(X_train_21, y_train_21)
            ccp_alphas = path.ccp_alphas
            
            step = max(1, len(ccp_alphas) // 20)
            alphas_to_test = ccp_alphas[::step]
            
            best_alpha = 0
            best_score = -np.inf
            
            for alpha in alphas_to_test:
                if alpha == 0:
                    continue
                dt_cv = DecisionTreeRegressor(ccp_alpha=alpha, random_state=42)
                scores = cross_val_score(dt_cv, X_train_21, y_train_21, 
                                        cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
                mean_score = scores.mean()
                
                if mean_score > best_score:
                    best_score = mean_score
                    best_alpha = alpha
            
            alpha_to_use = best_alpha if best_alpha > 0 else 0.0001
            
            dt_cv_pruned = DecisionTreeRegressor(ccp_alpha=alpha_to_use, random_state=42)
            dt_cv_pruned.fit(X_train_21, y_train_21)
            y_pred_21 = dt_cv_pruned.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_21):
                predictions_cv_pruned_21.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_21': pred
                })
    
    # Progress update
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(oos_dates)} dates ({100 * (i + 1) / len(oos_dates):.1f}%)")

print(f"\nCompleted.")
print(f"Expanding window predictions: {len(predictions_cv_pruned_exp):,}")
print(f"Rolling 252-day predictions: {len(predictions_cv_pruned_252):,}")
print(f"Rolling 21-day predictions: {len(predictions_cv_pruned_21):,}")

METHOD 5: CV-OPTIMIZED POST-PRUNED TREE
Processed 100/2948 dates (3.4%)
Processed 200/2948 dates (6.8%)
Processed 300/2948 dates (10.2%)
Processed 400/2948 dates (13.6%)
Processed 500/2948 dates (17.0%)
Processed 600/2948 dates (20.4%)
Processed 700/2948 dates (23.7%)
Processed 800/2948 dates (27.1%)
Processed 900/2948 dates (30.5%)
Processed 1000/2948 dates (33.9%)
Processed 1100/2948 dates (37.3%)
Processed 1200/2948 dates (40.7%)
Processed 1300/2948 dates (44.1%)
Processed 1400/2948 dates (47.5%)
Processed 1500/2948 dates (50.9%)
Processed 1600/2948 dates (54.3%)
Processed 1700/2948 dates (57.7%)
Processed 1800/2948 dates (61.1%)
Processed 1900/2948 dates (64.5%)
Processed 2000/2948 dates (67.8%)
Processed 2100/2948 dates (71.2%)
Processed 2200/2948 dates (74.6%)
Processed 2300/2948 dates (78.0%)
Processed 2400/2948 dates (81.4%)
Processed 2500/2948 dates (84.8%)
Processed 2600/2948 dates (88.2%)
Processed 2700/2948 dates (91.6%)
Processed 2800/2948 dates (95.0%)
Processed 2900/2948

In [18]:
# Merge and save Method 5 predictions
df_cv_pruned_exp = pd.DataFrame(predictions_cv_pruned_exp)
df_cv_pruned_252 = pd.DataFrame(predictions_cv_pruned_252)
df_cv_pruned_21 = pd.DataFrame(predictions_cv_pruned_21)

predictions_cv_pruned = df_cv_pruned_exp.merge(
    df_cv_pruned_252,
    on=['date', 'index'],
    how='outer'
).merge(
    df_cv_pruned_21,
    on=['date', 'index'],
    how='outer'
)

# Add symbol information
predictions_cv_pruned = predictions_cv_pruned.merge(
    data[['symbol']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder and sort
predictions_cv_pruned = predictions_cv_pruned[['date', 'symbol', 'index', 'pred_expanding', 'pred_rolling_252', 'pred_rolling_21']]
predictions_cv_pruned = predictions_cv_pruned.sort_values(['date', 'symbol']).reset_index(drop=True)

# Save predictions
OUTPUT_FILE_CV_PRUNED = os.path.join(OUTPUT_DATA_DIR, "predictions_decision_tree_cv_pruned.pkl")
predictions_cv_pruned.to_pickle(OUTPUT_FILE_CV_PRUNED)

print("\nMethod 5 - CV-Optimized Post-Pruned Tree Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_cv_pruned):,}")
print(f"Non-null predictions by window type:")
print(f"  Expanding: {predictions_cv_pruned['pred_expanding'].notna().sum():,}")
print(f"  Rolling 252-day: {predictions_cv_pruned['pred_rolling_252'].notna().sum():,}")
print(f"  Rolling 21-day: {predictions_cv_pruned['pred_rolling_21'].notna().sum():,}")
print(f"\nSaved to: {OUTPUT_FILE_CV_PRUNED}")
print(f"File size: {os.path.getsize(OUTPUT_FILE_CV_PRUNED) / (1024**2):.2f} MB")

# Display alpha selection statistics
if len(alpha_history) > 0:
    df_alpha = pd.DataFrame(alpha_history)
    print(f"\nAlpha Selection Statistics (Expanding Window):")
    print(f"  Mean alpha: {df_alpha['alpha'].mean():.6f}")
    print(f"  Median alpha: {df_alpha['alpha'].median():.6f}")
    print(f"  Min alpha: {df_alpha['alpha'].min():.6f}")
    print(f"  Max alpha: {df_alpha['alpha'].max():.6f}")


Method 5 - CV-Optimized Post-Pruned Tree Summary
Total observations: 3,001,267
Non-null predictions by window type:
  Expanding: 3,001,267
  Rolling 252-day: 3,001,267
  Rolling 21-day: 3,001,267

Saved to: C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/Data\predictions_decision_tree_cv_pruned.pkl
File size: 128.85 MB

Alpha Selection Statistics (Expanding Window):
  Mean alpha: 0.000000
  Median alpha: 0.000000
  Min alpha: 0.000000
  Max alpha: 0.000002
